# Load a CSV file into MySQL with R

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kartechbabu/MySQL_notebooks/blob/main/csv_to_mysql_r.ipynb)

**ITOM 6265 · Database Design for Business.** Explained in the book chapter
*Loading Data from Files* (Part III). Checked November 17, 2026 with R 4.3,
readr 2.1, DBI 1.2, RMariaDB 1.3 and MySQL 8.0.45. A Python version does the
same steps with pandas.

The plan: **read** the CSV file into a data frame, **connect** to your MySQL
database, and **append** the rows to a table with `dbAppendTable`.

**Your database account.** Never type your password into a code cell.

* **On your computer:** put `DB_HOST=...`, `DB_PORT=...`, `DB_USER=...`,
  `DB_PASSWORD=...` and `DB_NAME=...` in a `.env` file next to the notebook.
* **In Colab** (Runtime → Change runtime type → R): there is no `.env`, so the
  notebook asks for each value when it runs. R cannot hide what you type, so
  the notebook clears your answers, password included, from the output once
  it has connected.

## 1. Install and connect

In [1]:
# In Colab, install the MySQL driver once (it takes a minute or two).
if (!requireNamespace("RMariaDB", quietly = TRUE)) install.packages("RMariaDB")
library(DBI)
library(readr)

In [2]:
# On your computer: read DB_HOST, DB_USER, ... from a .env file, if there is one.
if (file.exists(".env")) readRenviron(".env")

# Find a setting; if it is missing (for example in Colab), ask for it.
setting <- function(name) {
  value <- Sys.getenv(name)
  if (value == "") value <- readline(paste0(name, ": "))
  value
}

# Connect. CLIENT_SSL encrypts the connection; RMariaDB does not by default.
con <- dbConnect(
  RMariaDB::MariaDB(),
  host     = setting("DB_HOST"),
  port     = as.integer(setting("DB_PORT")),
  user     = setting("DB_USER"),
  password = setting("DB_PASSWORD"),
  dbname   = setting("DB_NAME"),
  client.flag = RMariaDB::CLIENT_SSL,
  bigint = "integer"   # show COUNT(*) results as ordinary numbers
)
# Remove what you typed (including the password) from this cell's output.
IRdisplay::clear_output()

dbGetQuery(con, "SELECT DATABASE() AS my_database, VERSION() AS mysql_version")
# Check that the connection is encrypted (a cipher name, not empty).
dbGetQuery(con, "SHOW SESSION STATUS LIKE 'Ssl_cipher'")

my_database,mysql_version
<chr>,<chr>
0000_dem,8.0.45


Variable_name,Value
<chr>,<chr>
Ssl_cipher,TLS_AES_256_GCM_SHA384


## 2. Read the CSV file into a data frame

`read_csv` accepts a web address as well as a file name, so the notebook reads
the sample file straight from the course's GitHub repository. Addresses with
commas inside are wrapped in double quotes in the file; `read_csv` handles
that by default.

In [3]:
# The sample files live in the notebook's GitHub repository.
data_url <- "https://raw.githubusercontent.com/kartechbabu/MySQL_notebooks/main/data/"

df <- read_csv(paste0(data_url, "london_restaurants.csv"), show_col_types = FALSE)
dim(df)    # rows, columns
head(df)

[1] 15  9

name,city,address,longitude,latitude,average_cost,price_range,aggregate_rating,votes
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Gymkhana,London,"42 Albemarle Street, Mayfair, London W1S 4JH",-0.14164500,51.50852,80,4,4.7,214
Bocca Di Lupo,London,"12 Archer Street, Soho, London W1D 7BB",-0.13394700,51.51163,45,3,4.5,571
Flat Iron,London,"17 Beak Street, Soho, London W1F 9RW",-0.13834300,51.51207,35,2,4.9,309
Duck & Waffle,London,"Heron Tower, 110 Bishopsgate, City of London, London EC2N 4AY",-0.08096300,51.51628,55,3,4.9,706
Dishoom,London,"7 Boundary Street, Shoreditch, London E2 7JE",-0.07658056,51.52454,55,3,4.5,305
Yauatcha,London,"15-17 Broadwick Street, Soho, London W1F 0DL",-0.13522900,51.51374,90,4,4.7,1326


## 3. Case 1: append to a table that already exists

This is the everyday case: the table was designed in advance, and new data
arrives as a file. Here we create the table first, as its designer would:
real types, a **primary key**, and a `UNIQUE` rule so the same restaurant
cannot be loaded twice. (Running this cell again drops the table and starts
over with an empty one.)

In [4]:
invisible(dbExecute(con, "DROP TABLE IF EXISTS london_restaurants"))
invisible(dbExecute(con, "
CREATE TABLE london_restaurants (
  restaurant_id     INT           NOT NULL AUTO_INCREMENT PRIMARY KEY,
  name              VARCHAR(100)  NOT NULL,
  city              VARCHAR(50)   NOT NULL,
  address           VARCHAR(255)  NOT NULL,
  longitude         DECIMAL(9,6),
  latitude          DECIMAL(9,6),
  average_cost      DECIMAL(7,2),
  price_range       TINYINT,
  aggregate_rating  DECIMAL(2,1),
  votes             INT,
  CONSTRAINT london_restaurants_uq UNIQUE (name, address)
)"))
cat("table created\n")

table created


Now the load itself is one line. `dbAppendTable` adds the rows to the existing
table, matching data frame columns to table columns by name. It sends the
rows one at a time, so `dbWithTransaction` wraps them: if any row fails,
**all** of them are undone.

In [5]:
rows <- dbWithTransaction(con, dbAppendTable(con, "london_restaurants", df))
cat(rows, "rows loaded\n")

15 rows loaded


In [6]:
# Check: the table should hold as many rows as the file.
dbGetQuery(con, "SELECT restaurant_id, name, average_cost, votes
                 FROM london_restaurants ORDER BY restaurant_id LIMIT 5")

restaurant_id,name,average_cost,votes
<int>,<chr>,<dbl>,<int>
1,Gymkhana,80,214
2,Bocca Di Lupo,45,571
3,Flat Iron,35,309
4,Duck & Waffle,55,706
5,Dishoom,55,305


## 4. Case 2: a new table

**The recommended way** is the one above: write the `CREATE TABLE` yourself,
then append. You decide the types and the keys.

**The quick way** lets `dbWriteTable` create the table from the data frame.
On the class server this fails, because every table must have a primary key
and `dbWriteTable` does not create one:

In [7]:
tryCatch(
  dbWriteTable(con, "london_restaurants_quick", df),
  error = function(e) cat("Refused:", conditionMessage(e), "\n")
)

Refused: Unable to create or change a table without a primary key, when the system variable 'sql_require_primary_key' is set. Add a primary key to the table or unset this variable to avoid this message. Note that tables without a primary key can cause performance problems in row-based replication, so please consult your DBA before changing this setting. [3750] 


You *can* switch the primary-key rule off for this connection, let
`dbWriteTable` create the table, and add a key afterwards. Look at the types
it chose. `name` is `VARCHAR(24)`, just long enough for the longest name in
*this* file, so next week's file with a longer name would be refused. The
price, the rating and even the vote count are `DOUBLE`, an approximate number
type. That is why the recommended way is better.

In [8]:
invisible(dbExecute(con, "SET SESSION sql_require_primary_key = 0"))  # this connection only
dbWriteTable(con, "london_restaurants_quick", df)
invisible(dbExecute(con, "ALTER TABLE london_restaurants_quick
                          ADD restaurant_id INT NOT NULL AUTO_INCREMENT PRIMARY KEY FIRST"))
invisible(dbExecute(con, "SET SESSION sql_require_primary_key = 1"))

dbGetQuery(con, "SELECT column_name, column_type FROM information_schema.columns
                 WHERE table_schema = DATABASE() AND table_name = 'london_restaurants_quick'
                 ORDER BY ordinal_position")

COLUMN_NAME,COLUMN_TYPE
<chr>,<chr>
restaurant_id,int
name,varchar(24)
city,varchar(6)
address,varchar(61)
longitude,double
latitude,double
average_cost,double
price_range,double
aggregate_rating,double


**Never use `overwrite = TRUE` on a table you care about.** `dbWriteTable`
first **drops** the existing table, then tries to create a new one, which the
class server refuses. The result is no table at all:

In [9]:
tryCatch(
  dbWriteTable(con, "london_restaurants_quick", df, overwrite = TRUE),
  error = function(e) cat("Refused:", conditionMessage(e), "\n")
)
dbGetQuery(con, "SELECT COUNT(*) AS tables_named_quick FROM information_schema.tables
                 WHERE table_schema = DATABASE() AND table_name = 'london_restaurants_quick'")

Refused: Unable to create or change a table without a primary key, when the system variable 'sql_require_primary_key' is set. Add a primary key to the table or unset this variable to avoid this message. Note that tables without a primary key can cause performance problems in row-based replication, so please consult your DBA before changing this setting. [3750] 


tables_named_quick
<int>
0


## 5. A newer file: append, or update what is already there

`london_restaurants_update.csv` has five new restaurants and a newer row for
**Gymkhana**, which is already loaded. (Its vote count, 230, is invented to
show an update.) Appending fails on the duplicate, and the transaction undoes
the whole file:

In [10]:
df2 <- read_csv(paste0(data_url, "london_restaurants_update.csv"), show_col_types = FALSE)
tryCatch(
  dbWithTransaction(con, dbAppendTable(con, "london_restaurants", df2)),
  error = function(e) cat("Refused:", conditionMessage(e), "\n")
)
dbGetQuery(con, "SELECT COUNT(*) AS rows_in_table FROM london_restaurants")

Refused: Duplicate entry 'Gymkhana-42 Albemarle Street, Mayfair, London W1S 4JH' for key 'london_restaurants.london_restaurants_uq' [1062] 


rows_in_table
<int>
15


To insert the new rows **and** update the existing one, use an *upsert*:
`INSERT … ON DUPLICATE KEY UPDATE`. Write the SQL with `?` placeholders and
pass the data frame's columns as `params`; the statement runs once per row.
`AS new` names the incoming row.

In [11]:
upsert_sql <- "
INSERT INTO london_restaurants
  (name, city, address, longitude, latitude,
   average_cost, price_range, aggregate_rating, votes)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?) AS new
ON DUPLICATE KEY UPDATE votes = new.votes, aggregate_rating = new.aggregate_rating"

columns <- c("name", "city", "address", "longitude", "latitude",
             "average_cost", "price_range", "aggregate_rating", "votes")
invisible(dbWithTransaction(con, dbExecute(con, upsert_sql, params = unname(as.list(df2[columns])))))

dbGetQuery(con, "SELECT name, votes FROM london_restaurants
                 WHERE name IN ('Gymkhana', 'Bone Daddies')")

name,votes
<chr>,<int>
Bone Daddies,418
Gymkhana,230


In [12]:
dbGetQuery(con, "SELECT COUNT(*) AS rows_in_table FROM london_restaurants")

rows_in_table
<int>
20


## 6. A messy file: load as text, then convert in SQL

Real exports have stray spaces, empty cells and odd values. Load every column
**as text** into a staging table, so nothing is refused, then convert with one
`INSERT … SELECT`: `TRIM` removes spaces, `NULLIF(…, '')` turns empty cells
into `NULL`, and `CAST` sets the type.

In [13]:
raw <- read_csv(paste0(data_url, "london_restaurants.csv"),
                col_types = cols(.default = col_character()))  # every value as text

invisible(dbExecute(con, "DROP TABLE IF EXISTS london_restaurants_staging"))
invisible(dbExecute(con, "
CREATE TABLE london_restaurants_staging (
  row_id INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
  name VARCHAR(255), city VARCHAR(255), address VARCHAR(255),
  longitude VARCHAR(255), latitude VARCHAR(255), average_cost VARCHAR(255),
  price_range VARCHAR(255), aggregate_rating VARCHAR(255), votes VARCHAR(255))"))
cat(dbAppendTable(con, "london_restaurants_staging", raw), "rows staged\n")

15 rows staged


In [14]:
invisible(dbExecute(con, "DROP TABLE IF EXISTS london_restaurants_clean"))
invisible(dbExecute(con, "CREATE TABLE london_restaurants_clean LIKE london_restaurants"))  # typed target
converted <- dbExecute(con, "
INSERT INTO london_restaurants_clean
  (name, city, address, longitude, latitude,
   average_cost, price_range, aggregate_rating, votes)
SELECT TRIM(name), TRIM(city), TRIM(address),
       CAST(NULLIF(TRIM(longitude), '') AS DECIMAL(9,6)),
       CAST(NULLIF(TRIM(latitude), '') AS DECIMAL(9,6)),
       CAST(NULLIF(TRIM(average_cost), '') AS DECIMAL(7,2)),
       CAST(NULLIF(TRIM(price_range), '') AS UNSIGNED),
       CAST(NULLIF(TRIM(aggregate_rating), '') AS DECIMAL(2,1)),
       CAST(NULLIF(TRIM(votes), '') AS UNSIGNED)
FROM london_restaurants_staging
ORDER BY row_id")
cat(converted, "rows converted\n")
dbGetQuery(con, "SELECT name, average_cost, aggregate_rating, votes
                 FROM london_restaurants_clean ORDER BY restaurant_id LIMIT 3")

15 rows converted


name,average_cost,aggregate_rating,votes
<chr>,<dbl>,<dbl>,<int>
Gymkhana,80,4.7,214
Bocca Di Lupo,45,4.5,571
Flat Iron,35,4.9,309


## 7. Clean up

Drop the practice tables (set `CLEAN_UP <- FALSE` to keep them and explore).

In [15]:
CLEAN_UP <- TRUE
if (CLEAN_UP) {
  invisible(dbExecute(con, "DROP TABLE IF EXISTS london_restaurants_staging,
                            london_restaurants_quick, london_restaurants_clean, london_restaurants"))
  cat("practice tables dropped\n")
}
dbDisconnect(con)

practice tables dropped


## What to remember

* `read_csv` reads the file; `dbAppendTable` (inside `dbWithTransaction`)
  appends it to an existing table.
* Create new tables yourself (types, primary key, `UNIQUE` rules), then
  append. Never `overwrite = TRUE` on a table you need.
* Connect with `client.flag = RMariaDB::CLIENT_SSL`, and keep passwords out of
  the notebook.
* For a newer file, upsert with `INSERT … ON DUPLICATE KEY UPDATE`; for a
  messy file, stage as text and convert with `INSERT … SELECT`.